# Project #07 — ETL Progress Logs
**Raúl Cetina · UPY · Unit 2 · Commute & Well-Being**

---

## Research question
> *Is sun exposure during the commute linked to academic stress, burnout or sleep quality?*

**Approach for this checkpoint (behavioral proxy):** sun/heat exposure during the commute leaves a measurable trace on our shared platform: **hydration purchases** (water and electrolyte drinks) right after students arrive on campus. I cross those purchases with the **real weather in Mérida** (max temperature and UV index) to build the first indicators of the phenomenon. Stress/sleep variables will be added later through a short survey.

## Connected sources (3)
| # | Source | Technology | What it provides |
|---|--------|-----------|------------------|
| 1 | **Shared platform PostgreSQL** (Ágora Campus / UPY-Ecommerce, Neon) | `psycopg2` + SQL | Live purchases: orders, items, products |
| 2 | **Historical CSVs from the shared repo** (GitHub raw) | `pandas.read_csv` | Team sample catalog and orders |
| 3 | **Open-Meteo API** (Mérida, no API key) | `requests` | Daily max temperature and UV index |

## Progress completed at this checkpoint
- ✔ Connection to **2+ sources** of the shared platform (with visible logs)
- ✔ **Poller** inserting simulated purchases + incremental extraction via **polling** (simulated real time)
- ✔ **Cleaning**: timezone, duplicates, nulls, dtypes and text normalization
- ✔ **Indicator** tied to the question: *Morning Hydration Index (MHI, post-commute)* + daily correlation purchases↔UV/temperature
- ✔ **Visual ETL**: pipeline diagram and indicator charts

*This is a progress checkpoint, not a final product: limitations and next steps are listed at the end.*

*Note: product names and some data values are in Spanish because that is how they live in the team platform database.*

## 0 · Setup

In [ ]:
%pip install -q psycopg2-binary pandas matplotlib requests

import warnings
import psycopg2
import pandas as pd
import requests
import matplotlib.pyplot as plt
import unicodedata
import random, time, uuid
from datetime import datetime, timezone

# pandas warns that psycopg2 is not SQLAlchemy; good enough for this checkpoint.
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")

# --- Shared platform (Neon Postgres, pooler endpoint) ---
DB_URL = (
    "postgresql://neondb_owner:npg_Fr2iLGS8tZHm"
    "@ep-delicate-hat-atmk8wg6-pooler.c-9.us-east-1.aws.neon.tech/neondb"
    "?sslmode=require"
)
# --- CSVs from the team's shared repository ---
RAW = "https://raw.githubusercontent.com/DaniWin02/UPY-Ecommerce/main/data/samples/csv/"
# --- Mérida, Yucatán (for Open-Meteo) ---
LAT, LON = 20.97, -89.62
# UPY institutional colors for every chart (single series)
PURPLE, GOLD = "#4B2385", "#D9A33C"

print("Setup OK ·", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))
print("pandas", pd.__version__, "| psycopg2", psycopg2.__version__)

## 1 · EXTRACT — Source 1: platform PostgreSQL (live)
Direct connection to the team marketplace database. Connection log + row counts + preview of hydration purchases.

In [ ]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

cur.execute("select version()")
print("[connection 1 · OK] ", cur.fetchone()[0].split(" on ")[0])

for table in ("orders", "order_items", "products", "users"):
    cur.execute(f"select count(*) from {table}")
    print(f"    {table:<12} -> {cur.fetchone()[0]:>5} rows")

SQL_HYDRATION = """
    select o.id as order_id, o.created_at, o.estado as status, oi.cantidad as qty,
           oi.precio_unit as unit_price, p.nombre as product, o.referencia_pago as payment_ref
    from order_items oi
    join orders   o on o.id = oi.order_id
    join product_variants pv on pv.id = oi.variant_id
    join products p on p.id = pv.product_id
    where p.nombre ~* 'agua|electrolit|hidrat|suero'
      and o.estado in ('pago_verificado','preparando','listo_entrega','entregado')
    order by o.created_at
"""
df_db = pd.read_sql(SQL_HYDRATION, conn)
print(f"\n[extract 1] hydration purchases on the platform: {len(df_db)} rows")
df_db.tail(5)

## 2 · EXTRACT — Source 2: historical CSVs from the shared repo
Second platform source: the sample data published in the team repository (GitHub raw).

In [ ]:
df_prod_hist = pd.read_csv(RAW + "products.csv")
df_ord_hist  = pd.read_csv(RAW + "orders.csv")
print(f"[connection 2 · OK] products.csv -> {df_prod_hist.shape[0]} rows x {df_prod_hist.shape[1]} cols")
print(f"[connection 2 · OK] orders.csv   -> {df_ord_hist.shape[0]} rows x {df_ord_hist.shape[1]} cols")
print("\nHistorical catalog (sample):")
df_prod_hist.head(3)

## 3 · EXTRACT — Source 3: real Mérida weather (Open-Meteo)
Max temperature and UV index for the last 14 days — the **sun exposure** variable of my research question.

In [ ]:
resp = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": LAT, "longitude": LON,
        "daily": "temperature_2m_max,uv_index_max",
        "timezone": "America/Merida", "past_days": 14, "forecast_days": 1,
    },
    timeout=15,
)
print("[connection 3 · OK] Open-Meteo HTTP", resp.status_code)
raw = resp.json()["daily"]
df_weather = pd.DataFrame({
    "date": pd.to_datetime(raw["time"]).date,
    "temp_max": raw["temperature_2m_max"],
    "uv_max": raw["uv_index_max"],
})
print(f"[extract 3] {len(df_weather)} days of weather for Mérida")
df_weather.tail(5)

## 4 · Simulated real time: POLLER + extraction via POLLING
A **poller** inserts simulated purchases into the platform (probability weighted by local hour and today's real UV index) and, in the same cycle, the extractor **polls** the database reading only the rows newer than the last timestamp — the classic pattern of an incremental, near-real-time ETL.

*Simulated purchases are identifiable: buyer `etl.simulador@upy.edu.mx`, payment reference `SIM-…`, SKUs `ETL-AGUA-600` / `ETL-ELECTRO-625`.*

In [ ]:
# Fixed simulator references (resolved once)
cur.execute("select id from users where email = 'etl.simulador@upy.edu.mx'")
BUYER_ID = cur.fetchone()[0]
VARIANTS = {}
for sku in ("ETL-AGUA-600", "ETL-ELECTRO-625"):
    cur.execute("""select pv.id, coalesce(pv.precio_comunidad, pv.precio), p.vendor_id
                   from product_variants pv join products p on p.id = pv.product_id
                   where pv.sku = %s""", (sku,))
    vid, price, vendor_id = cur.fetchone()
    VARIANTS[sku] = (vid, float(price), vendor_id)

UV_TODAY = float(df_weather.iloc[-1]["uv_max"])
UV_FACTOR = min(UV_TODAY / 8.0, 1.5)
print(f"[poller] today's UV: {UV_TODAY:.1f} -> thirst factor {UV_FACTOR:.2f}")

HOUR_WEIGHT = {7:.9, 8:1, 9:.8, 10:.5, 11:.4, 12:.6, 13:.9, 14:.8, 15:.6, 16:.4, 17:.3, 18:.3}

def db_now():
    """Timestamp from the DATABASE clock.
    Postgres subtlety: now() is FROZEN at transaction start, so we use
    clock_timestamp() (real wall clock) and close the transaction with a
    commit so the next INSERT starts a fresh transaction (otherwise its
    created_at = now() would equal the start of THIS one)."""
    cur.execute("select clock_timestamp()")
    mark = cur.fetchone()[0]
    conn.commit()
    return mark

def insert_purchase():
    sku = random.choices(list(VARIANTS), weights=[0.7, 0.3])[0]
    vid, price, vendor_id = VARIANTS[sku]
    qty = random.choice([1, 1, 1, 2])
    ref = f"SIM-{uuid.uuid4().hex[:8].upper()}"
    cur.execute("""insert into orders (comprador_id, vendor_id, estado, total,
                     referencia_pago, metodo_entrega, aula)
                   values (%s,%s,'entregado',%s,%s,'punto','Cafetería UPY') returning id""",
                (BUYER_ID, vendor_id, price*qty, ref))
    oid = cur.fetchone()[0]
    cur.execute("insert into order_items (order_id, variant_id, cantidad, precio_unit) values (%s,%s,%s,%s)",
                (oid, vid, qty, price))
    cur.execute("""insert into payments (order_id, metodo, referencia, monto_declarado, estado, verificado_en)
                   values (%s,'efectivo',%s,%s,'verificado', now())""", (oid, ref, price*qty))
    conn.commit()
    return f"{qty}x {sku} (${price*qty:.2f})"

# --- poller + polling cycle (timestamps ALWAYS from the DB clock) ---
last_mark = db_now()
captured = 0
for cycle in range(1, 6):
    local_hour = (datetime.now(timezone.utc).hour - 6) % 24
    # minimum 50% probability during the demo so live captures are visible
    prob = max(min(HOUR_WEIGHT.get(local_hour, 0.2) * UV_FACTOR, 0.95), 0.5)
    if random.random() < prob:
        print(f"  [poller  {cycle}] inserts -> {insert_purchase()}")
    else:
        print(f"  [poller  {cycle}] no purchase this round (prob {prob:.0%})")
    time.sleep(2)
    new_rows = pd.read_sql(
        "select o.created_at, p.nombre, oi.cantidad from order_items oi "
        "join orders o on o.id = oi.order_id "
        "join product_variants pv on pv.id = oi.variant_id "
        "join products p on p.id = pv.product_id "
        "where o.created_at > %(ts)s and p.nombre ~* 'agua|electrolit'",
        conn, params={"ts": last_mark})
    conn.commit()  # close the read transaction (same reason as above)
    last_mark = db_now()
    captured += len(new_rows)
    print(f"  [polling {cycle}] +{len(new_rows)} new row(s) · running total {captured}")
print(f"\n[real time] the pipeline captured {captured} new purchases during the demo")

## 5 · TRANSFORM — dataset cleaning
Every operation prints its effect (before → after).

In [ ]:
# Re-read the FULL hydration dataset, now including the poller's inserts
df = pd.read_sql(SQL_HYDRATION, conn)
print(f"[cleaning 0] raw dataset: {df.shape[0]} rows x {df.shape[1]} cols")
print(df.dtypes.to_string(), "\n")

# 1) Exact duplicates
before = len(df)
df = df.drop_duplicates()
print(f"[cleaning 1] duplicates: {before} -> {len(df)} rows ({before - len(df)} removed)")

# 2) Nulls per column (policy: payment_ref may be null, key fields may not)
print("[cleaning 2] nulls per column:")
print(df.isna().sum().to_string())
df = df.dropna(subset=["created_at", "product", "qty", "unit_price"])
print(f"    after requiring key fields: {len(df)} rows")

# 3) Correct dtypes (Postgres numeric arrives as object/Decimal)
df["unit_price"] = df["unit_price"].astype(float)
df["qty"] = df["qty"].astype(int)
df["amount"] = df["unit_price"] * df["qty"]
print(f"[cleaning 3] dtypes: unit_price={df.unit_price.dtype}, qty={df.qty.dtype}, amount computed")

# 4) Timezone: UTC -> America/Merida (LOCAL hour is what matters for the commute)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["ts_local"] = df["created_at"].dt.tz_convert("America/Merida")
df["date"] = df["ts_local"].dt.date
df["hour"] = df["ts_local"].dt.hour
print(f"[cleaning 4] tz converted: {df['created_at'].iloc[-1]} UTC -> {df['ts_local'].iloc[-1]} Mérida")

# 5) Text normalization (lowercase, no accents -> stable categorization)
def normalize_text(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")
df["product_norm"] = df["product"].map(normalize_text)
# keywords stay in Spanish because product names live in Spanish in the DB
df["category"] = df["product_norm"].str.extract(r"(agua|electrolit|suero|hidrat)", expand=False).fillna("other")
print(f"[cleaning 5] categories detected: {df['category'].value_counts().to_dict()}")

print(f"\n[cleaning ✓] final dataset: {len(df)} clean rows")
df[["ts_local", "product", "category", "qty", "amount", "hour"]].tail(5)

## 6 · Indicators tied to the research question
**MHI — Morning Hydration Index (post-commute):** share of hydration purchases that happen in the 07:00–10:59 local window, right after the morning commute under the sun. Complement: daily correlation between purchases and UV/temperature.

In [ ]:
WINDOW = (7, 10)  # 07:00-10:59, arrival after the morning commute
df["post_commute"] = df["hour"].between(*WINDOW)

total_units = int(df["qty"].sum())
window_units = int(df.loc[df.post_commute, "qty"].sum())
mhi = window_units / total_units * 100 if total_units else 0.0

print("="*58)
print("  MHI (Morning Hydration Index, post-commute)")
print(f"  {window_units} of {total_units} units inside the 07-11h window -> MHI = {mhi:.1f}%")
print("="*58)

print("\nHydration units per local hour:")
per_hour = df.groupby("hour")["qty"].sum()
print(per_hour.to_string())

# Daily correlation with weather (days present in both sources)
per_day = df.groupby("date")["qty"].sum().rename("units").reset_index()
merged = per_day.merge(df_weather, on="date", how="inner")
print(f"\nDays with both purchases AND weather: {len(merged)}")
if len(merged) >= 3:
    print(f"  corr(units, uv_max)   = {merged.units.corr(merged.uv_max):+.3f}")
    print(f"  corr(units, temp_max) = {merged.units.corr(merged.temp_max):+.3f}")
else:
    print("  ⚠ Not enough overlapping days yet for a stable correlation;")
    print("    the poller keeps accumulating data during the week (checkpoint).")
merged.tail(5)

## 7 · Visual ETL
Pipeline diagram with the real volumes of this run, plus the indicator charts.

In [ ]:
# --- 7a: pipeline diagram (with real counts) ---
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis("off")

def box(x, y, w, h, title, detail, color=PURPLE):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="none", zorder=2))
    ax.text(x + w/2, y + h*0.62, title, ha="center", va="center",
            color="white", fontsize=10.5, fontweight="bold", zorder=3)
    ax.text(x + w/2, y + h*0.30, detail, ha="center", va="center",
            color="white", fontsize=8.5, zorder=3)

def arrow(x0, x1, y):
    ax.annotate("", xy=(x1, y), xytext=(x0, y),
                arrowprops=dict(arrowstyle="-|>", lw=2, color="#6b7280"))

box(0.00, 0.68, 0.235, 0.27, "EXTRACT · Postgres", f"{len(df_db)} live purchases")
box(0.00, 0.37, 0.235, 0.27, "EXTRACT · repo CSV", f"{len(df_prod_hist)} hist. products")
box(0.00, 0.06, 0.235, 0.27, "EXTRACT · Open-Meteo", f"{len(df_weather)} weather days")
box(0.38, 0.37, 0.24, 0.27, "TRANSFORM", f"{len(df)} clean rows · 5 ops")
box(0.76, 0.37, 0.24, 0.27, "INDICATOR", f"MHI = {mhi:.1f}%", color="#2e1355")

# Connectors: the 3 sources converge into TRANSFORM, then flow to INDICATOR.
arrow(0.245, 0.375, 0.505)
arrow(0.625, 0.755, 0.505)
ax.plot([0.245, 0.31, 0.31, 0.375], [0.815, 0.815, 0.505, 0.505], color="#6b7280", lw=2)
ax.plot([0.245, 0.31, 0.31, 0.375], [0.195, 0.195, 0.505, 0.505], color="#6b7280", lw=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("ETL Pipeline — Commute & Well-Being (checkpoint)", fontsize=12, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# --- 7b: hydration purchases per hour, with the post-commute window ---
hours = range(6, 21)
values = [int(per_hour.get(h, 0)) for h in hours]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axvspan(WINDOW[0] - 0.5, WINDOW[1] + 0.5, color=GOLD, alpha=0.18, zorder=1)
ax.bar(list(hours), values, color=PURPLE, width=0.72, zorder=2)
if max(values) > 0:
    imax = values.index(max(values))
    ax.text(list(hours)[imax], max(values) + 0.15, str(max(values)),
            ha="center", fontsize=9, fontweight="bold", color="#1f2937")
ax.text(sum(WINDOW)/2, ax.get_ylim()[1]*0.95, "post-commute window",
        ha="center", fontsize=8.5, color="#7a5b13")
ax.set_xlabel("Local hour (Mérida)"); ax.set_ylabel("Hydration units")
ax.set_title("When does campus hydrate? Purchases by hour", loc="left")
ax.set_xticks(list(hours)); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"Reading: {mhi:.1f}% of units are bought inside the 07-11h window (MHI).")

In [ ]:
# --- 7c: weather vs daily purchases (two panels, shared X — no dual axis) ---
if len(merged) >= 2:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                                 gridspec_kw={"height_ratios": [1, 1]})
    a1.plot(merged.date, merged.uv_max, color=PURPLE, marker="o", lw=2, label="Max UV")
    a1.plot(merged.date, merged.temp_max, color=GOLD, marker="s", lw=2, label="Max temp (°C)")
    a1.legend(frameon=False, fontsize=9); a1.set_ylabel("Exposure")
    a1.spines[["top", "right"]].set_visible(False)
    a1.set_title("Daily sun/heat vs hydration purchases", loc="left")
    a2.bar(merged.date, merged.units, color=PURPLE, width=0.6)
    a2.set_ylabel("Units"); a2.set_xlabel("Date")
    a2.spines[["top", "right"]].set_visible(False)
    fig.autofmt_xdate()
    plt.tight_layout(); plt.show()
else:
    print("Not enough overlapping purchase+weather days for this chart yet")
    print("(the poller will keep accumulating days — this is a checkpoint).")

## 8 · Checkpoint status and next steps

**Done at this checkpoint**
- 3 connections with visible logs (platform Postgres, repo CSVs, weather API).
- Poller + polling working: the pipeline captures new purchases live.
- 5 cleaning operations documented, each printing its effect.
- MHI indicator computed and visualized; first daily cross with UV/temperature.

**Honest limitations**
- Few overlapping purchase↔weather days so far: the correlation is not stable yet.
- Hydration purchases are a *proxy* for sun exposure, not a direct measure of stress/burnout/sleep.

**Next steps**
1. Keep the poller accumulating data for several days (it also runs as a standalone script: `etl/poller_agora.py`).
2. Short perceived-stress / sleep-quality survey to cross with the MHI per person.
3. Load the daily indicator into an `etl_indicators` table (full LOAD phase) and automate it.

---
*Session teardown: the database connection is released below.*

In [ ]:
cur.close()
conn.close()
print("[teardown] platform connection released. End of checkpoint.")